In [26]:
# This week
# 1, test pipeline with MIT-BIH dataset
# 2, figure out why AFDB dataset is not working

In [27]:
import meta
import load
import analysis
import preprocess
import numpy as np
import pandas as pd
from collections import defaultdict, Counter
np.random.seed(42)

In [28]:
# %cd yang/
# !pip install wfdb

In [29]:
isDrive = True
parent_dir = '/content/drive/MyDrive/git_repos/wfdb-python/yang/data' if isDrive else 'c:/Users/wuyan/Projects/wfdb-python/yang/data'

records = []
dataset = meta.Dataset(parent_dir, 'mitdb', 360, 'MLII')
records.extend(load.load_dataset(dataset))

  8%|▊         | 4/49 [00:00<00:06,  7.20it/s]

Error processing record 102-0: [Errno 2] No such file or directory: '/content/drive/MyDrive/git_repos/wfdb-python/yang/data/mitdb/102-0.hea'
Error processing record 102: "['MLII'] not in index"


 12%|█▏        | 6/49 [00:00<00:05,  8.22it/s]

Error processing record 104: "['MLII'] not in index"


100%|██████████| 49/49 [00:06<00:00,  7.86it/s]

Loaded 46 records from mitdb dataset.


In [30]:
# records = []

# dataset = meta.datasets[0]
# records.extend(load.load_dataset(dataset))

# dateset = meta.datasets[1]
# records.extend(load.load_dataset(dateset))

In [31]:
auxnote_override_map = defaultdict(lambda: "Other")
auxnote_override_map['(AFIB'] = "AF"
auxnote_override_map['(AFL'] = "AF"

analysis.print_records_annotations(records)
records = preprocess.remove_auxnote_by_identifier(records, start_id='(V')  # remove (V events
analysis.print_records_annotations(records)
#records = preprocess.filter_auxnote_by_identifier(records, start_id='(')  # keep only notations starts with (
#analysis.print_records_annotations(records)
records = preprocess.override_auxnote(records, auxnote_override_map)
analysis.print_records_annotations(records)

Counter({'(N': 506, 'MISSB': 428, '(B': 221, '(AFIB': 107, '(PREX': 103, '(T': 83, '(VT': 61, '(AFL': 45, '(NOD': 36, '(P': 34, '(SVTA': 26, '(VFL': 6, 'TS': 6, '(BII': 5, '(IVR': 4, 'PSE': 3, '(AB': 3, '(SBR': 1})
Counter({'(N': 506, 'MISSB': 428, '(B': 221, '(AFIB': 107, '(PREX': 103, '(T': 83, '(AFL': 45, '(NOD': 36, '(P': 34, '(SVTA': 26, 'TS': 6, '(BII': 5, '(IVR': 4, 'PSE': 3, '(AB': 3, '(SBR': 1})
Counter({'Other': 1459, 'AF': 152})


## Preprocess Data

In [32]:
records = preprocess.preprocess_records(records, frequency_resample=True, bandpass_filter=True)

100%|██████████| 46/46 [00:05<00:00,  8.21it/s]


## Generate Samples

In [ ]:
# stratify resample records based on AF label
# pct_test = 0.2
# from sklearn.model_selection import train_test_split
# record_with_af = np.array([record['df_ann']['AuxNote'].str.contains("AF").any() for record in records])
# records_train, records_test = train_test_split(records, stratify=record_with_af, test_size=pct_test, random_state=42)
# print(f"Train records size: {len(records_train)}")
# print(f"Test records size: {len(records_test)}")

# records_train_names = [record['record_name'] for record in records_train]
# records_test_names = [record['record_name'] for record in records_test]
# print(f"Train records names: {records_train_names}")
# print(f"Test records names: {records_test_names}")

In [33]:
# window_duration = 20  # seconds
# stride_duration = 5  # seconds

# auxnote_label_map = {"AF":1, "Other": 0}

# freq_to_keeprate_map_train = {1:0.1, 2:1}
# samples_train = preprocess.generate_samples_from_records(records_train,
#                                                     window_duration=window_duration, stride_duration=stride_duration,
#                                                     auxnote_label_map = auxnote_label_map, freq_to_keeprate_map = freq_to_keeprate_map_train)

# freq_to_keeprate_map_test = {1:1, 2:1}
# samples_test = preprocess.generate_samples_from_records(records_test,
#                                                     window_duration=window_duration, stride_duration=window_duration, # non overlapping
#                                                     auxnote_label_map = auxnote_label_map, freq_to_keeprate_map = freq_to_keeprate_map_test)



window_duration = 20  # seconds
stride_duration = 20  # seconds

auxnote_label_map = {"AF":1, "Other": 0}

freq_to_keeprate_map_train = {1:1, 2:1}
samples = preprocess.generate_samples_from_records(records,
                                                    window_duration=window_duration, stride_duration=stride_duration,
                                                    auxnote_label_map = auxnote_label_map, freq_to_keeprate_map = freq_to_keeprate_map_train)

100%|██████████| 4140/4140 [00:30<00:00, 134.99it/s]

Generated 4140 samples from 46 records.


In [34]:
samples[-1]

{'df_wave':                Time      MLII AuxNote
 356011  1780.060477  0.104069   Other
 356012  1780.065477  0.097104   Other
 356013  1780.070477  0.010558   Other
 356014  1780.075477 -0.078121   Other
 356015  1780.080477 -0.087739   Other
 ...             ...       ...     ...
 360006  1800.035539  0.142256   Other
 360007  1800.040539  0.199784   Other
 360008  1800.045539  0.253787   Other
 360009  1800.050539  0.297211   Other
 360010  1800.055539  0.313545   Other
 
 [4000 rows x 3 columns],
 'sample_frequency': 200,
 'data_name': 'mitdb',
 'record_name': '214',
 'df_label':     label
 0       0
 1       0
 2       0
 3       0
 4       0
 5       0
 6       0
 7       0
 8       0
 9       0
 10      0
 11      0
 12      0
 13      0
 14      0
 15      0
 16      0
 17      0
 18      0
 19      0,
 'labels': '0'}

In [35]:
# split the samples into train/test with 80/20
from sklearn.model_selection import train_test_split
samples_train, samples_test = train_test_split(samples, test_size=0.2, random_state=42)
print(f"Train samples size: {len(samples_train)}")
print(f"Test samples size: {len(samples_test)}")
print(f"Train samples labels: {Counter([sample['labels'] for sample in samples_train])}")
print(f"Test samples labels: {Counter([sample['labels'] for sample in samples_test])}")

Train samples size: 3312
Test samples size: 828
Train samples labels: Counter({'0': 2913, '1': 319, '01': 80})
Test samples labels: Counter({'0': 737, '1': 70, '01': 21})


In [36]:
def select_samples_by_label(samples, output_pct, output_size):
    samples_selected = []
    for label, pct in output_pct.items():
        num_samples = int(output_size * pct)
        label_samples = [sample for sample in samples if sample['labels'] == label]
        print(f"Resampling {len(label_samples)} samples to {num_samples} samples for label {label}")
        if len(label_samples) > num_samples:
            label_samples = np.random.choice(label_samples, size=num_samples, replace=False)
        elif len(label_samples) < num_samples:
            label_samples = np.random.choice(label_samples, size=num_samples, replace=True)
        samples_selected.extend(label_samples)
    return samples_selected

def print_sample_labels(samples):
    counter = Counter([sample['labels'] for sample in samples])
    print(f"Sample labels: {counter}")


In [37]:
print_sample_labels(samples_train)
print_sample_labels(samples_test)

Sample labels: Counter({'0': 2913, '1': 319, '01': 80})
Sample labels: Counter({'0': 737, '1': 70, '01': 21})


In [38]:
#pct_by_label = {'0':0.33, '1':0.33, '01':0.33}
pct_by_label = {'0':2/3, '1':1/3, '01':0.0}
num_train_samples = 310 * 3  #min(counter_train.values()) * 3
num_test_samples = 77 * 3   #min(counter_test.values()) * 3
samples_train_selected = select_samples_by_label(samples_train, pct_by_label, num_train_samples)
samples_test_selected = select_samples_by_label(samples_test, pct_by_label, num_test_samples)

# print the resampled label counters
# counter_train_selected = Counter([sample['labels'] for sample in samples_train_selected])
# counter_test_selected = Counter([sample['labels'] for sample in samples_test_selected])
# print(f"Selected train labels: {counter_train_selected}")
# print(f"Selected test labels: {counter_test_selected}")
print_sample_labels(samples_train_selected)
print_sample_labels(samples_test_selected)

Resampling 2913 samples to 620 samples for label 0
Resampling 319 samples to 310 samples for label 1
Resampling 80 samples to 0 samples for label 01
Resampling 737 samples to 154 samples for label 0
Resampling 70 samples to 77 samples for label 1
Resampling 21 samples to 0 samples for label 01
Sample labels: Counter({'0': 620, '1': 310})
Sample labels: Counter({'0': 154, '1': 77})


In [39]:
# num_classes = len(auxnote_label_map)
# X_train, Y_train_class = preprocess.generate_inputs_from_samples(samples_train_selected)
# X_test, Y_test_class = preprocess.generate_inputs_from_samples(samples_test_selected)
# Y_train = preprocess.class_to_onehot(Y_train_class, num_classes=num_classes)
# Y_test = preprocess.class_to_onehot(Y_test_class, num_classes=num_classes)
# print(f"Train set shape: {X_train.shape}, {Y_train.shape}")
# print(f"Test set shape: {X_test.shape}, {Y_test.shape}")

In [40]:
num_classes = len(auxnote_label_map)
X_train, Y_train_class = preprocess.generate_inputs_from_samples(samples_train_selected)
X_test, Y_test_class = preprocess.generate_inputs_from_samples(samples_test_selected)

def find_most_common_label(Y_train_class):
    most_common_labels = []
    for row in Y_train_class:
        counter = Counter(row)
        most_common_label = counter.most_common(1)[0][0]
        most_common_labels.append(most_common_label)
    return np.array(most_common_labels).reshape(-1, 1)

Y_train = preprocess.class_to_onehot(find_most_common_label(Y_train_class), num_classes=num_classes)
Y_test = preprocess.class_to_onehot(find_most_common_label(Y_test_class), num_classes=num_classes)

print(f"Train set shape: {X_train.shape}, {Y_train.shape}")
print(f"Test set shape: {X_test.shape}, {Y_test.shape}")

100%|██████████| 930/930 [00:00<00:00, 9544.12it/s]


Generated 930 samples
X shape: (930, 4000) y shape: (930, 20)


100%|██████████| 231/231 [00:00<00:00, 655.09it/s]

Generated 231 samples
X shape: (231, 4000) y shape: (231, 20)
Train set shape: (930, 4000), (930, 2)
Test set shape: (231, 4000), (231, 2)


## Training

In [42]:
from tensorflow.keras.layers import Input, Dense, Conv1D, GlobalAveragePooling1D, Dropout, BatchNormalization, ReLU
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import tensorflow as tf

def network(X_train, y_train, X_test, y_test,
             unit_size, class_size, class_weights,
             filter_sizes = {1:16, 2:32, 3:64}, kernel_sizes = {1:7, 2:5, 3:3}, dropout_rates={1:0.2, 2:0.2, 3:0.2},
             learning_rate=0.001, batch_size = 32, epochs=40,
             loss_type='cross_entropy', modelCheckPoint=False):

    ####################################################################
    output_size = unit_size * class_size  # output size is the number of classes times the number of units
    print("Class Weights:", class_weights)

    def custom_loss(y_true, y_pred):
        if class_size != len(class_weights):
            raise ValueError("The length of class_weights must match the number of classes.")

        # Reshape tensors
        y_true = tf.reshape(y_true, [-1, unit_size, class_size])
        y_pred = tf.reshape(y_pred, [-1, unit_size, class_size])

        loss = 0.0
        if loss_type == 'cross_entropy':
            for i in range(class_size):
                if class_weights[i] > 0:
                    loss += class_weights[i] * tf.reduce_mean(
                        -y_true[:, :, i] * tf.math.log(y_pred[:, :, i] + 1e-15) # average loss across all samples and units
                    )
        elif loss_type == 'mse':
            for i in range(class_size):
                if class_weights[i] > 0:
                    loss += class_weights[i] * tf.reduce_mean(
                        tf.square(y_true[:, :, i] - y_pred[:, :, i])  # average loss across all samples and units
                    )
        else:
            raise ValueError("Invalid loss type. Use 'cross_entropy' or 'mse'.")
        return loss

    def custom_accuracy(y_true, y_pred):
        y_true = tf.reshape(y_true, [-1, unit_size, class_size])
        y_pred = tf.reshape(y_pred, [-1, unit_size, class_size])
        correct_predictions = tf.equal(tf.argmax(y_true, axis=2), tf.argmax(y_pred, axis=2))
        return tf.reduce_mean(tf.cast(correct_predictions, tf.float32)) # average accuracy across all samples and units

    ####################################################################
    inputs_cnn = Input(shape=(X_train.shape[1], 1))

    # Block 1
    x = Conv1D(filter_sizes[1], kernel_size=kernel_sizes[1], padding='same')(inputs_cnn)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    x = Dropout(dropout_rates[1])(x)

    # Block 2
    x = Conv1D(filter_sizes[2], kernel_size=kernel_sizes[2], padding='same')(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    x = Dropout(dropout_rates[2])(x)
    x = Conv1D(filter_sizes[2], kernel_size=kernel_sizes[2], padding='same')(x)

    # Block 3
    x = BatchNormalization()(x)
    x = ReLU()(x)
    x = Conv1D(filter_sizes[3], kernel_size=kernel_sizes[3], padding='same')(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    x = Dropout(dropout_rates[3])(x)
    x = Conv1D(filter_sizes[3], kernel_size=kernel_sizes[3], padding='same')(x)

    # Final block
    x = BatchNormalization()(x)
    x = ReLU()(x)
    x = GlobalAveragePooling1D()(x)

    # Output layer
    outputs_cnn = Dense(output_size, activation='softmax')(x)

    optimizer = Adam(learning_rate=learning_rate)
    model = Model(inputs=inputs_cnn, outputs=outputs_cnn)
    #model.compile(optimizer=optimizer, loss=custom_loss, metrics=[custom_accuracy])
    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

    # define callbacks for early stopping and saving the best model with lowest validation loss
    callbacks = [EarlyStopping(monitor='val_loss', patience=10)]
    if modelCheckPoint:
        callbacks.append(ModelCheckpoint(filepath='best_model.h5', monitor='val_loss', save_best_only=True))
    history=model.fit(X_train, y_train, epochs=epochs,
                      callbacks=callbacks, batch_size=batch_size, validation_data=(X_test, y_test),
                      #class_weight=class_weights # assign class weights to the model
                      )
    if modelCheckPoint:
        model.load_weights('best_model.h5')
    return (model, history)

In [ ]:
model, history=network(X_train, Y_train, X_test, Y_test,
                       filter_sizes={1: 8, 2: 16, 3: 32},
                       kernel_sizes={1: 3, 2: 3, 3: 3},
                       dropout_rates={1: 0.3, 2: 0.3, 3: 0.3},
                       #unit_size=window_duration,
                       unit_size=1,
                       class_size=num_classes,
                       class_weights={0: 0.5, 1: 0.5},
                       learning_rate=0.0001,
                       #batch_size=32,
                       batch_size=256,
                       epochs=100,
                       loss_type='cross_entropy', modelCheckPoint=False)

Class Weights: {0: 0.5, 1: 0.5}
Epoch 1/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 40s 9s/step - accuracy: 0.3419 - loss: 0.8920 - val_accuracy: 0.6147 - val_loss: 0.6921
Epoch 2/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 33s 6s/step - accuracy: 0.3363 - loss: 0.8880 - val_accuracy: 0.4329 - val_loss: 0.6934
Epoch 3/100


In [ ]:
model.summary()

### Error Analysis

In [ ]:
analysis.plot_accuracy_and_loss(history, X_test, Y_test, model, accuracy_keys = ['accuracy', 'val_accuracy'])

In [ ]:
X_test.shape, Y_test.shape

In [ ]:
Y_pred_onehot = model.predict(X_test)
Y_pred_class = preprocess.onehot_to_class(Y_pred_onehot, num_classes=num_classes)
Y_test_onehot = Y_test.copy()
Y_test_class = preprocess.onehot_to_class(Y_test_onehot, num_classes=num_classes)

# calculate the accuracy for each class
for i in range(num_classes):
    acc = np.sum((Y_test_class == i) & (Y_pred_class == i)) / np.sum(Y_test_class == i)
    print(f"Accuracy for class {i}: {acc:.2f}")

In [ ]:
# calculate the confusion matrix based on Y_test_class and Y_pred_class
C = np.zeros((num_classes, num_classes))
for i in range(len(Y_test_class)):
    for j in range(len(Y_test_class[i])):
        C[int(Y_test_class[i][j]), int(Y_pred_class[i][j])] += 1
print("Confusion Matrix:")
print(C)